# 🔬 Notebook 3: Notification System — Deep Dive: Priority, Retries, Idempotency

## 🛠️ Setup

```bash
cd 06-system-designs/notification-system
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Deep dive 1 — priority queues with starvation protection

In [ ]:
import heapq, itertools, random
from collections import deque

# Simple fix-ratio scheduler: pull 4 high : 2 normal : 1 low in each round.
class PriorityScheduler:
    def __init__(self):
        self.queues = {"high": deque(), "normal": deque(), "low": deque()}
        self.ratio  = {"high": 4, "normal": 2, "low": 1}

    def push(self, level, msg): self.queues[level].append(msg)

    def drain_round(self):
        out = []
        for level, n in self.ratio.items():
            for _ in range(n):
                if self.queues[level]:
                    out.append((level, self.queues[level].popleft()))
        return out

s = PriorityScheduler()
for i in range(10): s.push("low", f"promo-{i}")
for i in range(5):  s.push("normal", f"txn-{i}")
for i in range(3):  s.push("high", f"2fa-{i}")

# Drain until empty, show the interleaving
while any(s.queues.values()):
    print(s.drain_round())


## Deep dive 2 — retries with exponential backoff + DLQ

External providers *will* fail. Naive retry = thundering herd. Instead:

- On failure, requeue with delay = `min(cap, base * 2^attempt) + jitter`.
- After N attempts, drop to a **dead-letter queue** for human review.


In [ ]:
import random, time

def backoff(attempt, base=0.1, cap=10.0):
    # exponential with full jitter
    sleep = min(cap, base * (2 ** attempt))
    return random.uniform(0, sleep)

MAX_ATTEMPTS = 5
def try_send(msg):
    # Pretend to fail 50% of the time
    if random.random() < 0.5: return "FAIL"
    return "OK"

random.seed(1)
dlq = []
for msg in ["m1","m2","m3"]:
    for attempt in range(MAX_ATTEMPTS):
        res = try_send(msg)
        if res == "OK":
            print(f"{msg} sent on attempt {attempt+1}")
            break
        wait = backoff(attempt)
        print(f"  {msg} attempt {attempt+1} failed, waiting {wait:.3f}s")
    else:
        dlq.append(msg)
        print(f"{msg} → DLQ after {MAX_ATTEMPTS} attempts")
print("DLQ:", dlq)


## Deep dive 3 — idempotency & dedup keys

Why not just hash the payload?
- Same payload sent intentionally twice (e.g., resend receipt) would be swallowed.
- Dedup keys are **caller-assigned meaning**: "this is order-123-shipped, you've seen it."

Store `dedup_key → {status, first_seen_ts, last_ts}` in a fast KV (Redis). TTL long enough
to cover any retry window (e.g., 7 days).


In [ ]:
# Tiny dedup store
import time
store: dict[str, dict] = {}
def enqueue_with_dedup(key: str, payload: dict):
    if key in store:
        store[key]["last_ts"] = time.time()
        return "DUPLICATE, not re-sent"
    store[key] = {"payload": payload, "status": "queued", "first_seen_ts": time.time()}
    return "NEW, queued"

print(enqueue_with_dedup("order-123-shipped", {"m":"Your order shipped!"}))
print(enqueue_with_dedup("order-123-shipped", {"m":"Your order shipped!"}))  # swallowed
print(enqueue_with_dedup("order-124-shipped", {"m":"Another order!"}))
